# PPE training — Colab GPU
GPU çalışma zamanını seçin. Yerelde `python -m scripts.colab_bundle` ile oluşturulan `ppe-colab-source.zip` dosyasını aşağıdaki hücrede yükleyin. Veri, ağırlık ve gizli ayarlar pakete dahil değildir.


In [ ]:
%pip install ultralytics==8.4.183


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
root = Path("/content/PPE")
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.namelist():
        assert (root / member).resolve().is_relative_to(root), "Unsafe archive path"
    bundle.extractall(root)
%cd /content/PPE
%pip install -e ".[test]"


In [ ]:
import torch
assert torch.cuda.is_available(), "GPU çalışma zamanını seçin"
print(torch.cuda.get_device_name(0))


## İncelenmiş veri sürümünü yükle
Yerelde oluşturulan ve verify_release kontrolünü geçen veri sürümünün ZIP dosyasını yükleyin. Eski Construction-PPE kaynağı artık otomatik eğitime alınmaz. Pilot yalnız baret/baş/kişi içerir; yelek ve nihai test başarısı iddiası taşımaz.


In [ ]:
import subprocess, sys, yaml
release_upload = files.upload()
release_zip = next(name for name in release_upload if name.endswith(".zip"))
release_root = Path("/content/PPE/datasets/reviewed-pilot")
from scripts.prepare_shel5k import safe_extract
safe_extract(Path(release_zip), release_root)
config_path = release_root / "data.yaml"
config = yaml.safe_load(config_path.read_text())
config["path"] = str(release_root)  # Only relocate the local dataset root.
config_path.write_text(yaml.safe_dump(config))
from scripts.verify_release import verify_release
verification = verify_release(release_root)
assert verification["passed"], verification["errors"]
print(verification)
data_path = str(config_path)


## Kısa deneme
3 epoch yalnızca eğitim yolunun çalıştığını sınar. Üretilen train_batch ve val_batch görsellerini kontrol edin.


In [ ]:
subprocess.run([sys.executable, "-m", "scripts.train", "--data", data_path, "--device", "0", "--epochs", "3", "--name", "ppe-smoke"], check=True)


## Pilot sınırı
Bu küçük sürüm en fazla 3 epoch eğitim hattı denemesine izin verir. Baret örnekleri ve sahne kapsamı artırılmadan uzun eğitim veya model başarısı karşılaştırması yapılmaz.


## Sonuçları yedekle
Son hücreler Google Drive izni ister ve eğitim çıktılarıyla paket sürümlerini saklar. Oturum kapanmadan yerel kopya da indirin.


In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
run = Path("runs/train/ppe-smoke")
assert (run / "weights/best.pt").exists()
(run / "environment.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
archive = shutil.make_archive("/content/ppe-pilot-smoke", "zip", run)
shutil.copy2(archive, "/content/drive/MyDrive/ppe-pilot-smoke.zip")
files.download(archive)
